# Batch Scoring — Account Churn Prediction

Scores every account; writes predictions + per-industry churn risk summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/churn_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'mrr_usd', 'seat_count', 'tenure_days', 'health_score',
    'user_count', 'active_user_count', 'avg_logins_30d',
    'event_count', 'distinct_features', 'avg_duration',
    'ticket_count', 'sla_breach_count', 'avg_csat', 'avg_resolution_hrs',
]
cat_cols = ['plan', 'industry', 'region']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('churn_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_churn', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('churn_probability') > 0.8, 'critical')
        .when(col('churn_probability') > 0.6, 'high')
        .when(col('churn_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'account_id', 'plan', 'industry', 'region',
        'mrr_usd', 'health_score', 'avg_logins_30d', 'avg_csat',
        'had_churn', 'predicted_churn', 'churn_probability', 'risk_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-industry churn risk summary
summary = (
    predictions
    .groupBy('industry')
    .agg(
        count('*').alias('total_accounts'),
        spark_sum('predicted_churn').alias('predicted_churn_count'),
        spark_sum('had_churn').alias('actual_churn_count'),
        spark_round(avg('churn_probability'), 4).alias('avg_churn_probability'),
        spark_round(spark_sum('mrr_usd'), 0).alias('total_mrr'),
        spark_round(avg('health_score'), 1).alias('avg_health_score'),
    )
    .withColumn('churn_rate', spark_round(col('predicted_churn_count') / col('total_accounts') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_churn_probability') > 0.6, 'high')
        .when(col('avg_churn_probability') > 0.3, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_churn_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Industry churn summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')